# Chapter 6: Integer Mastery for VLSI Automation

Companion notebook to **python-from-zero** · `lesson-06` · based on *Python for VLSI*, Chapter 6.

### You will learn

- Use integers to count gates, violations, and banks.
- Decimal, binary, octal, and hex literals fluently.
- Bitwise operations for masks, flags, and memory addressing.
- Safe integer arithmetic with no silent overflow.

In [1]:
import sys, platform
print(f"Python {sys.version.split()[0]} on {platform.system()}")
assert sys.version_info >= (3, 9), "This notebook needs Python 3.9 or newer."

Python 3.12.3 on Linux


## Integer basics

Python integers are arbitrary precision — no overflow. Underscores help readability for large counts.

In [2]:
gate_count   = 1_250_430
die_mm2      = 12
gates_per_mm = gate_count // die_mm2
print(gate_count, gates_per_mm, 2**128)


1250430 104202 340282366920938463463374607431768211456


## Bases and conversions

`0b`, `0o`, `0x` for literals; `bin()`, `oct()`, `hex()` to print.

In [3]:
addr = 0x0040
mask = 0b1111_0000
print(bin(addr), oct(addr), hex(addr))
print("mask =", bin(mask), "=", mask)


0b1000000 0o100 0x40
mask = 0b11110000 = 240


## Bitwise operations

`&` and, `|` or, `^` xor, `~` not, `<<` shift left, `>>` shift right. Used constantly for register fields and address maps.

In [4]:
flags = 0b0000_0000
READ, WRITE, BURST = 1<<0, 1<<1, 1<<2

flags |= READ | BURST
print("flags       :", bin(flags))
print("burst set?  :", bool(flags & BURST))

flags &= ~READ
print("after clear :", bin(flags))


flags       : 0b101
burst set?  : True
after clear : 0b100


## Bank addressing

Extract bank index from a memory address via shifts and masks.

In [5]:
def bank_of(addr, bank_bits=2, row_bits=8):
    return (addr >> row_bits) & ((1 << bank_bits) - 1)

for a in (0x000, 0x100, 0x200, 0x3FF):
    print(f"addr=0x{a:04X}  bank={bank_of(a)}")


addr=0x0000  bank=0
addr=0x0100  bank=1
addr=0x0200  bank=2
addr=0x03FF  bank=3


## Integer vs float division

`/` always returns float. `//` floors toward negative infinity.

In [6]:
for pair in [(7, 2), (-7, 2), (7, -2)]:
    a, b = pair
    print(f"{a}/{b}={a/b:+.2f}  {a}//{b}={a//b:+d}  {a}%{b}={a%b:+d}")


7/2=+3.50  7//2=+3  7%2=+1
-7/2=-3.50  -7//2=-4  -7%2=+1
7/-2=-3.50  7//-2=-4  7%-2=-1


## Parsing integers from reports

`int(s, base)` reads a string in the given base. Great for addresses.

In [7]:
tokens = ["0x1A", "0b1010", "42", "0o17"]
for t in tokens:
    if t.startswith("0x"):
        v = int(t, 16)
    elif t.startswith("0b"):
        v = int(t, 2)
    elif t.startswith("0o"):
        v = int(t, 8)
    else:
        v = int(t)
    print(f"{t:8s} -> {v}")


0x1A     -> 26
0b1010   -> 10
42       -> 42
0o17     -> 15


## Population count (popcount)

Counting set bits is a classic VLSI kernel — compare the textbook loop with Python's built-in `int.bit_count`.

In [8]:
def popcount(x):
    c = 0
    while x:
        c += x & 1
        x >>= 1
    return c

for v in (0, 7, 255, 0xDEAD):
    print(f"{v:#06x}  manual={popcount(v)}  builtin={v.bit_count()}")


0x0000  manual=0  builtin=0
0x0007  manual=3  builtin=3
0x00ff  manual=8  builtin=8
0xdead  manual=11  builtin=11


## VLSI practice — register field packer

Pack three fields (mode, bank, offset) into a single 16-bit register and unpack them back.

In [9]:
def pack(mode, bank, offset):
    assert 0 <= mode   < 1<<4
    assert 0 <= bank   < 1<<4
    assert 0 <= offset < 1<<8
    return (mode << 12) | (bank << 8) | offset

def unpack(reg):
    return {
        "mode":   (reg >> 12) & 0xF,
        "bank":   (reg >> 8)  & 0xF,
        "offset": reg & 0xFF,
    }

reg = pack(mode=0xB, bank=0x3, offset=0x5A)
print(f"reg = 0x{reg:04X}")
print("unpack:", unpack(reg))


reg = 0xB35A
unpack: {'mode': 11, 'bank': 3, 'offset': 90}


### Exercise 1
Print the binary, octal, and hex of 255.

In [10]:
# Your code here


<details><summary>Show solution</summary>

```python
v=255; print(bin(v), oct(v), hex(v))
```

</details>

### Exercise 2
Return True if bit 3 of `flags = 0b1010` is set.

In [11]:
# Your code here


<details><summary>Show solution</summary>

```python
flags=0b1010; print(bool(flags & (1<<3)))
```

</details>

### Exercise 3
Write `clear_bit(x, n)` that clears bit `n` in `x`.

In [12]:
# Your code here


<details><summary>Show solution</summary>

```python
def clear_bit(x,n): return x & ~(1<<n)
print(bin(clear_bit(0b1111, 2)))
```

</details>

## Recap

- Python ints are unbounded — no overflow surprises.
- Bitwise ops (`&|^~<<>>`) and masks are the backbone of register code.
- Use `int(s, base)` to parse hex/bin strings from reports.
- `int.bit_count()` replaces hand-rolled popcount loops.

## What's next

Continue with **Chapter 7: String Mastery for VLSI Professionals** in this repo, and the matching lesson on [python-from-zero](https://python-from-zero.pages.dev).